# Dask

Dask 适合把 Pandas/NumPy 风格的数据计算扩展到多核或多机，也适合构建延迟执行图。它和 Ray 都能做分布式计算，但 Dask 更偏数据分析和任务图。


In [ ]:
from dask import delayed
from dask.distributed import Client, LocalCluster, as_completed


def load_order(order_id: int) -> dict[str, int]:
    return {'order_id': order_id, 'amount': order_id * 100}


def discount(order: dict[str, int]) -> int:
    return int(order['amount'] * 0.9)


with LocalCluster(n_workers=2, threads_per_worker=1, dashboard_address=None) as cluster:
    with Client(cluster) as client:
        orders = [delayed(load_order)(order_id) for order_id in range(1, 6)]
        amounts = [delayed(discount)(order) for order in orders]
        total = delayed(sum)(amounts)
        print('Delayed Total:', total.compute())

        futures = client.map(load_order, range(6, 9))
        for future in as_completed(futures):
            print('Future Result:', future.result())


## 远程集群

| 角色 | IP 地址 |
| --- | --- |
| Scheduler | 192.168.1.10 |
| Worker 1 | 192.168.1.11 |
| Worker 2 | 192.168.1.12 |

Scheduler 节点运行：

```zsh
dask-scheduler
```

Worker 节点运行：

```zsh
dask-worker tcp://192.168.1.10:8786
```

客户端连接：

```python
from dask.distributed import Client

client = Client('tcp://192.168.1.10:8786')
```
